# Full Production Simulation & Analysis Pipeline Notebook

This notebook mirrors the production pipeline script `scripts/run_full_production.sh` 1-to-1.
It executes the complete 200 MeV electron injector linac optimization campaign across:
- **Phase 1**: Scalarized Bayesian Optimization (`SingleTaskGP` / `qLogNEI`)
- **Phase 2**: Unconstrained Multi-Objective Bayesian Optimization (`qLogNEHVI`)
- **Phase 3**: Constraint-Aware Multi-Objective Bayesian Optimization (`qLogNEHVI` with explicit GP constraint models)
- **Comparative Analysis & Independent Pareto Rerun Audit**
- **Engineering Tolerance Robustness Analysis**

In [ ]:
# Step 1: Setup Environment and Paths
import os
from pathlib import Path
import sys
import torch

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.environ["ASTRA_BIN"] = str(project_root / "bin" / "astra")
os.environ["GENERATOR_BIN"] = str(project_root / "bin" / "generator")

print(f"Project Root: {project_root}")
print(f"ASTRA Binary: {os.environ['ASTRA_BIN']}")

In [ ]:
# Step 2: Run Phase 1 Scalarized BO Simulation
from mobo_linac.cli import run_scalarized
import argparse

args_p1 = argparse.Namespace(
    config=str(project_root / "configs" / "mobo_200MeV.yaml"),
    n_iterations=5,
    batch_size=4,
    num_initial_samples=16,
    num_workers=4,
    weights=[1.0, 1.0, 1.0],
    seed=42,
    output_dir=str(project_root / "results" / "full_production" / "phase1_scalarized"),
)
print("Starting Phase 1 Scalarized BO...")
run_scalarized(args_p1)

In [ ]:
# Step 3: Run Phase 2 Unconstrained MOBO Simulation
from mobo_linac.campaigns.runner import MoboCampaignRunner

runner_p2 = MoboCampaignRunner(
    config=project_root / "configs" / "mobo_200MeV.yaml",
    run_name="phase2_unconstrained",
    output_dir=project_root / "results" / "full_production" / "phase2_unconstrained",
    num_initial_samples=16,
    num_batches=5,
    batch_size=4,
    num_workers=4,
    seed=42,
    acq_type="qLogNEHVI",
    constrained=False,
)
res_p2, tracker_p2, dir_p2 = runner_p2.run()

In [ ]:
# Step 4: Run Phase 3 Constraint-Aware MOBO Simulation
runner_p3 = MoboCampaignRunner(
    config=project_root / "configs" / "mobo_200MeV.yaml",
    run_name="phase3_constrained",
    output_dir=project_root / "results" / "full_production" / "phase3_constrained",
    num_initial_samples=16,
    num_batches=5,
    batch_size=4,
    num_workers=4,
    seed=42,
    acq_type="qLogNEHVI",
    constrained=True,
)
res_p3, tracker_p3, dir_p3 = runner_p3.run()

In [ ]:
# Step 5: Comparative Analysis and Pareto Audit
from scripts.run_comparison_and_verification import generate_comparison_report

analysis_dir = project_root / "results" / "full_production" / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
print(f"Comparative analysis outputs written to: {analysis_dir}")

In [ ]:
# Step 6: Engineering Tolerance Robustness Analysis
from scripts.run_robustness_analysis import run_robustness_analysis
import argparse

args_rob = argparse.Namespace(
    pareto_csv=str(dir_p3 / "pareto.csv"),
    config=str(project_root / "configs" / "mobo_200MeV.yaml"),
    output_dir=str(analysis_dir / "robustness"),
    num_workers=4,
    num_perturbations=10,
    seed=42,
)
run_robustness_analysis(args_rob)